In [ ]:
!pip install pandas transformers torch sacrebleu nltk sentencepiece

In [ ]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')

In [ ]:

import pandas as pd
file_id = "1egAMS6QLuEdALUq0SaR8S1C3Z0iciPtc"
csv_url = f"https://drive.google.com/uc?export=download&id={file_id}"

df = pd.read_csv(csv_url)
df.head()

In [ ]:
import os

import torch
from tqdm import tqdm
import nltk
import sacrebleu
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# =====================================================================
print("[INFO] Initializing NLTK Environment...")
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

possible_paths = ["/root/nltk_data", "/usr/share/nltk_data", "/usr/local/share/nltk_data", "/usr/lib/nltk_data", "/usr/local/lib/nltk_data"]
for p in possible_paths:
    if p not in nltk.data.path:
        nltk.data.path.append(p)


from nltk.translate.meteor_score import meteor_score

In [ ]:

def translate_mbart50(src_texts, batch_size=16, device="cuda"):
    model_name = "facebook/mbart-large-50-many-to-many-mmt"
    tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
    model = MBartForConditionalGeneration.from_pretrained(model_name).to(device)
    tokenizer.src_lang = "zh_CN"
    forced_bos_token_id = tokenizer.lang_code_to_id["en_XX"]

    translated_results = []
    for i in tqdm(range(0, len(src_texts), batch_size), desc="mBART-50 Translating"):
        batch = src_texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            generated_tokens = model.generate(**inputs, forced_bos_token_id=forced_bos_token_id, max_length=128, num_beams=4, early_stopping=True)
        translated_results.extend(tokenizer.batch_decode(generated_tokens, skip_special_tokens=True))
    return translated_results

def translate_nllb(src_texts, batch_size=16, device="cuda"):
    model_name = "facebook/nllb-200-distilled-600M"
    tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="zho_Hans")
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    forced_bos_token_id = tokenizer.convert_tokens_to_ids("eng_Latn")

    translated_results = []
    for i in tqdm(range(0, len(src_texts), batch_size), desc="NLLB-200 Translating"):
        batch = src_texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            generated_tokens = model.generate(**inputs, forced_bos_token_id=forced_bos_token_id, max_length=128, num_beams=4, early_stopping=True)
        translated_results.extend(tokenizer.batch_decode(generated_tokens, skip_special_tokens=True))
    return translated_results


def calculate_lightweight_metrics(hypotheses, references):
    results = {}

    # 1. BLEU Score
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    results['BLEU'] = bleu.score

    # 2. ChrF Score
    chrf = sacrebleu.corpus_chrf(hypotheses, [references])
    results['ChrF'] = chrf.score

    # 3. METEOR Score 
    meteor_scores = []
    for hyp, ref in zip(hypotheses, references):
        try:
            hyp_tokens = nltk.word_tokenize(hyp)
            ref_tokens = nltk.word_tokenize(ref)
            meteor_scores.append(meteor_score([ref_tokens], hyp_tokens))
        except Exception as e:
            
            meteor_scores.append(0.0)

    results['METEOR'] = (sum(meteor_scores) / len(meteor_scores)) * 100 if meteor_scores else 0
    return results


if __name__ == "__main__":
  
    #CSV_FILE_PATH = "gold_manually_200_en.csv"  
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[INFO] Running on platform device: {DEVICE}")

    
    #df = pd.read_csv(CSV_FILE_PATH)
    src_sentences = df['text'].astype(str).tolist()
    ref_sentences = df['english'].astype(str).tolist()

   
    df['mbart50_translation'] = translate_mbart50(src_sentences, batch_size=16, device=DEVICE)
    df['nllb_translation'] = translate_nllb(src_sentences, batch_size=16, device=DEVICE)

  
    mbart_metrics = calculate_lightweight_metrics(df['mbart50_translation'].tolist(), ref_sentences)
    nllb_metrics = calculate_lightweight_metrics(df['nllb_translation'].tolist(), ref_sentences)

   
    print("\n" + "="*50 + "\n         EVALUATION REPORT (WITHOUT COMET)\n" + "="*50)
    metrics_df = pd.DataFrame([mbart_metrics, nllb_metrics], index=['mBART-50', 'NLLB-200'])
    print(metrics_df.to_string())
    print("="*50)

    
    df.to_csv("clean_evaluation_results.csv", index=False, encoding='utf-8-sig')
    print("[SUCCESS] Process completed without errors.")